# ОНС 5 Lite — версия на PyTorch

Задание сохранено по смыслу: используется тот же датасет писателей, те же параметры экспериментов и та же идея модели — `Embedding + Conv1D + MaxPooling + Dense`.

Главное отличие: обучение переведено с TensorFlow/Keras на PyTorch.

# **Импорты**

In [ ]:
import os
import re
import time
import zipfile
from collections import Counter

import gdown
import matplotlib.pyplot as plt
import numpy as np

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline

# Фиксируем генераторы случайных чисел, чтобы результаты были стабильнее
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство для обучения:', DEVICE)


# **Загрузка данных**

In [ ]:
# Скачивание и распаковка архива с текстами писателей
DATA_URL = 'https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip'
ARCHIVE_NAME = 'writers.zip'
DATA_FOLDER = 'writers'

gdown.download(DATA_URL, ARCHIVE_NAME, quiet=True)

os.makedirs(DATA_FOLDER, exist_ok=True)

with zipfile.ZipFile(ARCHIVE_NAME, 'r') as archive:
    archive.extractall(DATA_FOLDER)

print('Данные загружены и распакованы.')


# **Пути до файлов**

In [ ]:
# Основные настройки для чтения файлов датасета
FILE_DIR = 'writers'
SIG_TRAIN = 'обучающая'
SIG_TEST = 'тестовая'


# **Подготовка списков и добавление текстов**

In [ ]:
# Списки с названиями классов и текстами для обучения/проверки
CLASS_LIST = []
text_train = []
text_test = []

# Перебираем все файлы в папке датасета
for file_name in os.listdir(FILE_DIR):
    match = re.match(r'\((.+)\) (\S+)_', file_name)

    if match is None:
        continue

    class_name = match.group(1)
    subset_name = match.group(2).lower()

    is_train_file = SIG_TRAIN in subset_name
    is_test_file = SIG_TEST in subset_name

    if not (is_train_file or is_test_file):
        continue

    if class_name not in CLASS_LIST:
        CLASS_LIST.append(class_name)
        text_train.append('')
        text_test.append('')
        print(f'Добавление класса "{class_name}"')

    class_index = CLASS_LIST.index(class_name)
    print(f'Добавление файла "{file_name}" в класс "{CLASS_LIST[class_index]}", {subset_name} выборка.')

    file_path = os.path.join(FILE_DIR, file_name)
    with open(file_path, 'r', encoding='utf-8') as file:
        file_text = file.read().replace('\n', ' ')

    if is_train_file:
        text_train[class_index] += ' ' + file_text
    else:
        text_test[class_index] += ' ' + file_text


# **Определение количества классов и вывод текстов**

In [ ]:
# Количество классов определяется после загрузки текстов
CLASS_COUNT = len(CLASS_LIST)

print(CLASS_LIST)
print('Количество классов:', CLASS_COUNT)

# Контрольный вывод показываем первые символы каждого класса
for class_index, class_name in enumerate(CLASS_LIST):
    print(f'Класс: {class_name}')
    print(f'  train: {text_train[class_index][:200]}')
    print(f'  test : {text_test[class_index][:200]}')
    print()


# **Контекстный менеджер**

In [ ]:
# Небольшой менеджер для замера времени выполнения блока кода
class timex:
    def __enter__(self):
        self.start_time = time.time()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        elapsed_time = time.time() - self.start_time
        print('Время обработки: {:.2f} с'.format(elapsed_time))


# **Токенизация текста без Keras**

In [ ]:
# Простой токенизатор, заменяющий tensorflow.keras.preprocessing.text.Tokenizer.
# Он приводит текст к нижнему регистру, удаляет служебные символы и оставляет самые частотные слова.
FILTERS = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'


class SimpleTokenizer:
    def __init__(self, num_words, filters=FILTERS, lower=True):
        self.num_words = num_words
        self.filters = filters
        self.lower = lower
        self.word_index = {}

    def _split_text(self, text):
        if self.lower:
            text = text.lower()

        replace_table = str.maketrans({symbol: ' ' for symbol in self.filters})
        text = text.translate(replace_table)
        return text.split()

    def fit_on_texts(self, texts):
        word_counter = Counter()
        first_position = {}

        for text in texts:
            for word in self._split_text(text):
                if word not in first_position:
                    first_position[word] = len(first_position)
                word_counter[word] += 1

        sorted_words = sorted(
            word_counter,
            key=lambda word: (-word_counter[word], first_position[word])
        )

        # Индекс 0 оставляем техническим, как это обычно делается в Embedding.
        self.word_index = {
            word: index + 1
            for index, word in enumerate(sorted_words)
        }

    def texts_to_sequences(self, texts):
        sequences = []

        for text in texts:
            encoded_text = []
            for word in self._split_text(text):
                word_id = self.word_index.get(word)

                # Оставляем только слова, попавшие в ограничение словаря.
                if word_id is not None and word_id < self.num_words:
                    encoded_text.append(word_id)

            sequences.append(encoded_text)

        return sequences


# **Разбиение текста на окна**

In [ ]:
# Нарезка последовательностей на окна фиксированной длины.
# Метки классов остаются целыми числами, потому что PyTorch CrossEntropyLoss работает именно так.
def get_samples(texts, win_size, win_hop):
    samples = []
    labels = []

    for class_index, token_sequence in enumerate(texts):
        last_start = len(token_sequence) - win_size + 1

        for start in range(0, last_start, win_hop):
            end = start + win_size
            samples.append(token_sequence[start:end])
            labels.append(class_index)

    x_data = np.array(samples, dtype=np.int64)
    y_data = np.array(labels, dtype=np.int64)

    return x_data, y_data


# **Модель на PyTorch**

In [ ]:
class TextConvNet(nn.Module):
    def __init__(self, vocab_size, win_size, class_count, embedding_dim=50):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0,
        )

        self.spatial_dropout = nn.Dropout1d(p=0.2)
        self.batch_norm = nn.BatchNorm1d(num_features=embedding_dim)

        self.conv = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=20,
            kernel_size=5,
        )

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.dropout = nn.Dropout(p=0.2)

        conv_length = win_size - 5 + 1
        pooled_length = conv_length // 2
        linear_input = 20 * pooled_length

        self.classifier = nn.Linear(linear_input, class_count)

    def forward(self, x):
        # x: [размер_пакета, длина_окна]
        x = self.embedding(x)

        # Для Conv1d нужен формат [размер_пакета, каналы, длина]
        x = x.permute(0, 2, 1)

        x = self.spatial_dropout(x)
        x = self.batch_norm(x)

        x = self.conv(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.dropout(x)

        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)

        # Softmax здесь не нужен: CrossEntropyLoss принимает сырые выходы модели.
        return x


# **Функции обучения и проверки**

In [ ]:
def make_loader(x_data, y_data, batch_size=128, shuffle=False):
    x_tensor = torch.tensor(x_data, dtype=torch.long)
    y_tensor = torch.tensor(y_data, dtype=torch.long)

    dataset = TensorDataset(x_tensor, y_tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
    )


def run_epoch(model, data_loader, criterion, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_objects = 0

    for x_batch, y_batch in data_loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)

            if is_training:
                loss.backward()
                optimizer.step()

        batch_size = y_batch.size(0)
        predictions = outputs.argmax(dim=1)

        total_loss += loss.item() * batch_size
        total_correct += (predictions == y_batch).sum().item()
        total_objects += batch_size

    mean_loss = total_loss / total_objects
    mean_accuracy = total_correct / total_objects

    return mean_loss, mean_accuracy


def train_model(model, train_loader, val_loader, epochs=20, learning_rate=0.001, verbose=False):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = {
        'loss': [],
        'accuracy': [],
        'val_loss': [],
        'val_accuracy': [],
    }

    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = run_epoch(
            model=model,
            data_loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
        )

        with torch.no_grad():
            val_loss, val_accuracy = run_epoch(
                model=model,
                data_loader=val_loader,
                criterion=criterion,
                optimizer=None,
            )

        history['loss'].append(train_loss)
        history['accuracy'].append(train_accuracy)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_accuracy)

        if verbose:
            print(
                f'Эпоха {epoch:02d}/{epochs} | '
                f'loss={train_loss:.4f}, acc={train_accuracy:.4f} | '
                f'val_loss={val_loss:.4f}, val_acc={val_accuracy:.4f}'
            )

    return history


# **Графики обучения**

In [ ]:
def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    axes[0].plot(history['accuracy'], label='Обучающая выборка')
    axes[0].plot(history['val_accuracy'], label='Проверочная выборка')
    axes[0].set_title('График точности')
    axes[0].set_xlabel('Эпоха')
    axes[0].set_ylabel('Точность')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history['loss'], label='Обучающая выборка')
    axes[1].plot(history['val_loss'], label='Проверочная выборка')
    axes[1].set_title('График ошибки')
    axes[1].set_xlabel('Эпоха')
    axes[1].set_ylabel('Ошибка')
    axes[1].legend()
    axes[1].grid(True)

    plt.show()


# **Функция одного эксперимента**

In [ ]:
def run_experiment(vocab_size, win_size, win_hop, epochs=20, batch_size=128):
    tokenizer = SimpleTokenizer(num_words=vocab_size)
    tokenizer.fit_on_texts(text_train)

    train_sequences = tokenizer.texts_to_sequences(text_train)
    test_sequences = tokenizer.texts_to_sequences(text_test)

    x_train, y_train = get_samples(train_sequences, win_size, win_hop)
    x_test, y_test = get_samples(test_sequences, win_size, win_hop)

    print('Размер обучающей выборки:', x_train.shape)
    print('Размер проверочной выборки:', x_test.shape)

    train_loader = make_loader(
        x_data=x_train,
        y_data=y_train,
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = make_loader(
        x_data=x_test,
        y_data=y_test,
        batch_size=batch_size,
        shuffle=False,
    )

    model = TextConvNet(
        vocab_size=vocab_size,
        win_size=win_size,
        class_count=CLASS_COUNT,
    ).to(DEVICE)

    with timex():
        history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=epochs,
            learning_rate=0.001,
            verbose=False,
        )

    plot_training_history(history)

    best_accuracy = max(history['val_accuracy'])
    print('Лучшая точность на проверочной выборке:', round(best_accuracy, 4))

    return best_accuracy


# **Проведение первого эксперимента**

In [ ]:
results = []

# Проверяем влияние размера словаря при фиксированных параметрах окна
for vocab_size in [5000, 10000, 20000, 40000]:
    print(f'\nТестирование VOCAB_SIZE = {vocab_size}...')

    best_accuracy = run_experiment(
        vocab_size=vocab_size,
        win_size=1000,
        win_hop=100,
    )

    results.append({
        'VOCAB_SIZE': vocab_size,
        'WIN_SIZE': 1000,
        'WIN_HOP': 100,
        'Accuracy': best_accuracy,
    })


# **Проведение второго эксперимента**

In [ ]:
# Проверяем влияние размера окна и шага при фиксированном размере словаря
for win_size, win_hop in [(500, 50), (2000, 200)]:
    print(f'\nТестирование WIN_SIZE = {win_size}, WIN_HOP = {win_hop}...')

    best_accuracy = run_experiment(
        vocab_size=20000,
        win_size=win_size,
        win_hop=win_hop,
    )

    results.append({
        'VOCAB_SIZE': 20000,
        'WIN_SIZE': win_size,
        'WIN_HOP': win_hop,
        'Accuracy': best_accuracy,
    })


# **Сводная таблица**

In [ ]:
# Итоговый вывод результатов всех запусков
header = '{:<12} | {:<10} | {:<10} | {:<10}'.format(
    'VOCAB_SIZE',
    'WIN_SIZE',
    'WIN_HOP',
    'Accuracy',
)

print('\n' + header)
print('-' * 50)

for row in results:
    print('{:<12} | {:<10} | {:<10} | {:<10.4f}'.format(
        row['VOCAB_SIZE'],
        row['WIN_SIZE'],
        row['WIN_HOP'],
        row['Accuracy'],
    ))
